# 02 - DATA PREPROCESSING

**Mục tiêu:** Biến đổi dữ liệu thô từ PostgreSQL thành dữ liệu sạch đảm bảo tính nhất quán, hợp lệ và sẵn sàng cho phân tích khám phá dữ liệu (EDA) và mô hình hóa.

## Mục lục

- [Import](#import)

- [Khởi tạo & Kết nối dữ liệu](#khoi-tao-va-ket-noi-du-lieu)

- [Tiền xử lý các bảng](#tien-xu-ly-cac-bang)

    - [1. Bảng Category](#section-1-bang-category)
    - [2. Bảng Seller](#section-2-bang-seller)
    - [3. Bảng Product](#section-3-bang-product)
    - [4. Bảng Price Offer](#section-4-bang-price-offer)
    - [5. Bảng Review](#section-5-bang-review)
    - [6. Bảng Reviewer](#section-6-bang-reviewer)
    - [7. Bảng Service](#section-7-bang-service)
    - [8. Bảng Coupon](#section-8-bang-coupon)
    - [9. Bảng Offer Coupon](#section-9-bang-offer-coupon)
    - [10. Bảng Offer Service](#section-10-bang-offer-service)

- [Lưu trữ dữ liệu](#luu-tru-du-lieu)

## Import

In [5]:
import sys
import os

root_path = os.path.abspath(os.path.join('..'))
if root_path not in sys.path:
    sys.path.append(root_path)

import polars as pl
from sqlalchemy import create_engine
from dotenv import load_dotenv

import warnings
warnings.filterwarnings('ignore')

<h2 id="khoi-tao-va-ket-noi-du-lieu">Khởi tạo và kết nối dữ liệu</h2>

Sử dụng `SQLAlchemy` để kết nối cơ sở dữ liệu và `dotenv` để bảo mật thông tin tài khoản. Dữ liệu được tải thông qua thư viện `Polars` nhằm tối ưu hiệu suất đọc và xử lý theo hướng đa luồng.

### Cấu hình kết nối
Các biến môi trường như `LOCAL_DB_USER`, `LOCAL_DB_PASS`, `LOCAL_DB_HOST`, `LOCAL_DB_NAME` và `LOCAL_DB_PORT` được nạp từ file môi trường để tạo chuỗi kết nối PostgreSQL một cách linh hoạt và an toàn.

In [6]:
load_dotenv()

USER = os.getenv("LOCAL_DB_USER")
PASSWORD = os.getenv("LOCAL_DB_PASS")
HOST = os.getenv("LOCAL_DB_HOST")
DBNAME = os.getenv("LOCAL_DB_NAME")
PORT = os.getenv("LOCAL_DB_PORT")

DATABASE_URL = f"postgresql://{USER}:{PASSWORD}@{HOST}:{PORT}/{DBNAME}"
engine = create_engine(DATABASE_URL)

### Tải dữ liệu từ PostgreSQL
Sau khi khởi tạo `engine`, toàn bộ 10 bảng dữ liệu được truy vấn trực tiếp từ PostgreSQL bằng `pl.read_database(...)`. Việc tách dữ liệu theo từng thực thể giúp quá trình làm sạch dễ kiểm soát hơn trước khi đồng bộ quan hệ giữa các bảng.

In [7]:
with engine.connect() as conn:
    df_category = pl.read_database("SELECT * FROM category", conn)
    df_seller = pl.read_database("SELECT * FROM seller", conn)
    df_product = pl.read_database("SELECT * FROM product", conn)
    df_price_offer = pl.read_database("SELECT * FROM price_offer", conn)
    df_review = pl.read_database("SELECT * FROM review", conn)
    df_reviewer = pl.read_database("SELECT * FROM reviewer", conn)
    df_service = pl.read_database("SELECT * FROM service", conn)
    df_coupon = pl.read_database("SELECT * FROM coupon", conn)
    df_offer_coupon = pl.read_database("SELECT * FROM offer_coupon", conn)
    df_offer_service = pl.read_database("SELECT * FROM offer_service", conn)

In [8]:
RAW_PATH = "../data/raw"

df_category.write_parquet(f"{RAW_PATH}/category.parquet")
df_seller.write_parquet(f"{RAW_PATH}/seller.parquet")
df_product.write_parquet(f"{RAW_PATH}/product.parquet")
df_price_offer.write_parquet(f"{RAW_PATH}/price_offer.parquet")
df_review.write_parquet(f"{RAW_PATH}/review.parquet")
df_reviewer.write_parquet(f"{RAW_PATH}/reviewer.parquet")
df_service.write_parquet(f"{RAW_PATH}/service.parquet")
df_coupon.write_parquet(f"{RAW_PATH}/coupon.parquet")
df_offer_coupon.write_parquet(f"{RAW_PATH}/offer_coupon.parquet")
df_offer_service.write_parquet(f"{RAW_PATH}/offer_service.parquet")

<h2 id="tien-xu-ly-cac-bang">TIỀN XỬ LÝ CÁC BẢNG</h2>

<h3 id="section-1-bang-category">1. Bảng Category</h3>

In [9]:
print(f"Kích thước bảng Category: {df_category.shape}")
print(f"Kiểu dữ liệu của bảng Category:\n{df_category.dtypes}")

print("Kiểm tra số lượng dữ liệu null trong bảng Category:")
display(df_category.null_count())

display(df_category.head())

Kích thước bảng Category: (3901, 7)
Kiểu dữ liệu của bảng Category:
[String, String, Null, Int64, String, String, Boolean]
Kiểm tra số lượng dữ liệu null trong bảng Category:


category_id,category_name,parent_category_id,level,category_path,category_url,is_scanned
u32,u32,u32,u32,u32,u32,u32
0,0,3901,0,0,0,0


category_id,category_name,parent_category_id,level,category_path,category_url,is_scanned
str,str,null,i64,str,str,bool
"""1883""","""Nhà Cửa - Đời Sống""",null,1,"""Nhà Cửa - Đời Sống""","""https://tiki.vn/nha-cua-doi-so…",true
"""1789""","""Điện Thoại - Máy Tính Bảng""",null,1,"""Điện Thoại - Máy Tính Bảng""","""https://tiki.vn/dien-thoai-may…",true
"""2567""","""Đồ dùng vệ sinh cho bé""",null,3,"""Đồ Chơi - Mẹ & Bé > Đồ dùng ch…","""https://tiki.vn/do-dung-ve-sin…",true
"""2576""","""Dụng cụ chăm sóc sức khỏe""",null,3,"""Đồ Chơi - Mẹ & Bé > Đồ dùng ch…","""https://tiki.vn/dung-cu-cham-s…",true
"""2593""","""Đồ dùng đi chơi""",null,3,"""Đồ Chơi - Mẹ & Bé > Đồ dùng ch…","""https://tiki.vn/do-dung-di-cho…",true


In [10]:
df_category = df_category.unique(subset=["category_id"], keep='first') \
            .drop('is_scanned') \
            .with_columns([
                pl.col("category_id").cast(pl.Int64),
                pl.col("parent_category_id").cast(pl.Int64),
                pl.col("level").cast(pl.Int8),
                pl.col("category_path").str.strip_chars(),
                pl.col("category_name").str.strip_chars(),
                pl.col("category_url").str.strip_chars()
            ]) \
            .with_columns([
                pl.col('parent_category_id').fill_null(0)
            ]) \
            .sort(by=['level', 'category_id'])

In [11]:
print(f"Kích thước bảng Category sau khi làm sạch: {df_category.shape}")
display(df_category.head())

Kích thước bảng Category sau khi làm sạch: (3901, 6)


category_id,category_name,parent_category_id,level,category_path,category_url
i64,str,i64,i8,str,str
915,"""Thời trang nam""",0,1,"""Thời trang nam""","""https://tiki.vn/thoi-trang-nam…"
931,"""Thời trang nữ""",0,1,"""Thời trang nữ""","""https://tiki.vn/thoi-trang-nu/…"
976,"""Túi thời trang nữ""",0,1,"""Túi thời trang nữ""","""https://tiki.vn/tui-vi-nu/c976"""
1520,"""Làm Đẹp - Sức Khỏe""",0,1,"""Làm Đẹp - Sức Khỏe""","""https://tiki.vn/lam-dep-suc-kh…"
1686,"""Giày - Dép nam""",0,1,"""Giày - Dép nam""","""https://tiki.vn/giay-dep-nam/c…"


<h3 id="section-2-bang-seller">2. Bảng Seller</h3>

Với bảng `Seller`, dữ liệu được làm sạch bằng cách loại bỏ trùng lặp theo `seller_id`, chuẩn hóa chuỗi ở `seller_name`, điền giá trị mặc định cho các trường còn thiếu và tối ưu kiểu dữ liệu số để giảm chi phí lưu trữ.

In [12]:
print(f"Kích thước bảng Seller: {df_seller.shape}")
print(f"Kiểu dữ liệu của bảng Seller:\n{df_seller.dtypes}")

print("Kiểm tra số lượng dữ liệu null trong bảng Seller:")
display(df_seller.null_count())
display(df_seller.head())

Kích thước bảng Seller: (4705, 4)
Kiểu dữ liệu của bảng Seller:
[String, String, Float64, Int64]
Kiểm tra số lượng dữ liệu null trong bảng Seller:


seller_id,seller_name,seller_rating,total_reviews
u32,u32,u32,u32
0,0,0,0


seller_id,seller_name,seller_rating,total_reviews
str,str,f64,i64
"""sach-tieng-nhat-jlpt""","""SÁCH TIẾNG NHẬT JLPT""",5.0,1
"""noi-that-nha-minh""","""NỘI THẤT ADORA NEMADORA""",4.8,32
"""bookcity-vn""","""BOOKCITY.VN""",4.8,1200
"""espp""","""ESPP""",4.5,117
"""2hbooks""","""2HBooks""",4.9,168


In [13]:
df_seller = df_seller.unique(subset=["seller_id"], keep="first")\
        .with_columns([
            pl.col("seller_id").str.strip_chars(),
            pl.col("seller_name").str.strip_chars()
        ])\
        .with_columns([
            pl.col("seller_rating").cast(pl.Float32),
            pl.col("total_reviews").cast(pl.Int32)
        ])\
        .with_columns([
            pl.col("seller_name").fill_null("Đang cập nhật"),
            pl.col("seller_rating").fill_null(0.0),
            pl.col("total_reviews").fill_null(0)
        ])

In [14]:
print(f"Kích thước bảng Seller sau khi làm sạch: {df_seller.shape}")
display(df_seller.head())

Kích thước bảng Seller sau khi làm sạch: (4705, 4)


seller_id,seller_name,seller_rating,total_reviews
str,str,f32,i32
"""hieu-sach-nhoc-miko""","""Hiệu sách Nhóc Miko""",4.3,30
"""boneco-viet-nam""","""BONECO Official Store""",4.6,118
"""phu-kien-legend""","""Phụ Kiện Legend""",4.4,2800
"""soc-brothers""","""Soc&Brothers""",4.8,613
"""kanstorevn""","""Kanstorevn""",0.0,0


<h3 id="section-3-bang-product">3. Bảng Product</h3>

Ngoài việc loại bỏ trùng lặp theo `product_id`, notebook còn chuẩn hóa tên sản phẩm, mô tả ngắn, thương hiệu/tác giả và các trường số liệu tổng hợp để giảm sai lệch trong các bước EDA sau này.

In [15]:
print(f"Kích thước bảng Product: {df_product.shape}")
print(f"Kiểu dữ liệu của bảng Product:\n{df_product.dtypes}")

print("Kiểm tra số lượng dữ liệu null trong bảng Product:")
display(df_product.null_count())
display(df_product.head())

Kích thước bảng Product: (310110, 11)
Kiểu dữ liệu của bảng Product:
[String, String, String, String, String, String, String, String, Int64, Int64, Float64]
Kiểm tra số lượng dữ liệu null trong bảng Product:


product_id,product_name,short_description,category_id,seller_id,product_url,image_url,author_brand,sold_quantity,review_count,review_score
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
0,0,134682,0,433,0,0,2526,0,0,0


product_id,product_name,short_description,category_id,seller_id,product_url,image_url,author_brand,sold_quantity,review_count,review_score
str,str,str,str,str,str,str,str,i64,i64,f64
"""275822950""","""Ốp lưng dẻo trong dành cho Hua…",null,"""28584""","""atzstore""","""https://tiki.vn/op-lung-deo-tr…","""https://salt.tikicdn.com/cache…","""GREENCASE""",0,0,0.0
"""273982699""","""Ốp lưng dẻo trong dành cho Hua…",null,"""28584""","""phu-kien-bingo""","""https://tiki.vn/op-lung-deo-tr…","""https://salt.tikicdn.com/cache…","""ULTRA THIN""",0,0,0.0
"""275236637""","""Luyện Thi Năng Lực Nhật Ngữ Tr…","""N/A""","""67878""","""tien-tho-book-hn""","""https://tiki.vn/luyen-thi-nang…","""https://salt.tikicdn.com/cache…","""""",0,0,0.0
"""273454695""","""2500 Từ Vựng Cần Thiết Cho Kỳ …","""N/A""","""67878""","""info-book""","""https://tiki.vn/2500-tu-vung-c…","""https://salt.tikicdn.com/cache…","""ARC ACADEMY""",0,0,0.0
"""273423214""","""Giáo Trình Luyện Thi Năng Lực …","""N/A""","""67878""","""nhan-van""","""https://tiki.vn/giao-trinh-luy…","""https://salt.tikicdn.com/cache…","""NHÓM TÁC GIẢ""",6,0,0.0


In [16]:
promo_pattern = r"(?i)(mua\s*\d|giảm\s*\d|giảm\s*k|giảm\s*%|tặng|freeship|hoàn tiền|voucher|đơn từ|rẻ hơn|sale|combo|ưu đãi|giá gốc)"

df_product = df_product.unique(subset=["product_id"], keep="first")\
            .with_columns([
                pl.col("product_id").cast(pl.Int64),
                pl.col("category_id").cast(pl.Int64),
                pl.col("seller_id").str.strip_chars(),
                pl.col("product_name").str.strip_chars(),
                pl.col("author_brand").str.strip_chars(),
                pl.col("short_description").str.strip_chars().replace("N/A", None),
                pl.col("image_url").str.strip_chars()
            ])\
            .with_columns(
                pl.when(pl.col("author_brand").str.contains(promo_pattern))
                .then(pl.lit(None))
                .otherwise(pl.col("author_brand"))
                .alias("author_brand")
            )\
            .with_columns(
                pl.col("author_brand")
                .str.replace(r"(?i)^(0|null|n/a|na|none)$", "")
                .replace("", None)
            )\
            .with_columns([
                pl.col("sold_quantity").cast(pl.Int32).fill_null(0),
                pl.col("review_count").cast(pl.Int32).fill_null(0),
                pl.col("review_score").cast(pl.Float32).fill_null(0.0)
            ])

In [17]:
print(f"Kích thước bảng Product sau khi làm sạch: {df_product.shape}")
df_product.head()

Kích thước bảng Product sau khi làm sạch: (310110, 11)


product_id,product_name,short_description,category_id,seller_id,product_url,image_url,author_brand,sold_quantity,review_count,review_score
i64,str,str,i64,str,str,str,str,i32,i32,f32
278968043,"""Khăn phủ đàn piano cơ đàn pian…",null,23450,"""hoang-hai-store""","""https://tiki.vn/khan-phu-dan-p…","""https://salt.tikicdn.com/cache…","""SUZIN""",0,0,0.0
34819234,"""Bài học Israel cuộc hồi sinh v…",null,9725,"""info-book""","""https://tiki.vn/bai-hoc-israel…","""https://salt.tikicdn.com/cache…","""NGUYỄN HIẾN LÊ""",0,0,0.0
2012301,"""Hộp thực phẩm vuông 1600ml Ino…",null,67548,"""bach-loc-toan-gia""","""https://tiki.vn/hop-thuc-pham-…","""https://salt.tikicdn.com/cache…","""INOMATA""",16,1,5.0
275618982,"""Bao da rút IONECASE điện thoại…",null,28594,"""opbavn""","""https://tiki.vn/bao-da-rut-ion…","""https://salt.tikicdn.com/cache…","""IONECASE""",0,0,0.0
101870772,"""Comics Will Break Your Heart""",null,9422,"""nha-sach-fahasa""","""https://tiki.vn/comics-will-br…","""https://salt.tikicdn.com/cache…","""FAITH ERIN HICKS""",1,0,0.0


<h3 id="section-4-bang-price-offer">4. Bảng Price Offer</h3>

Bảng `Price Offer` ghi nhận thông tin giá theo thời điểm crawl. Quá trình làm sạch tập trung vào loại bỏ bản ghi trùng, chuẩn hóa định danh `offer_id`, `product_id`, kiểm soát giá trị rỗng và ép kiểu các trường giá về dạng số thực để sẵn sàng cho phân tích biến động giá.

In [18]:
print(f"Kích thước bảng Price_Offer: {df_price_offer.shape}")
print(f"Kiểu dữ liệu của bảng Price_Offer:\n{df_price_offer.dtypes}")

print("Kiểm tra số lượng dữ liệu null trong bảng Price_Offer:")
display(df_price_offer.null_count())
display(df_price_offer.head())

Kích thước bảng Price_Offer: (1027622, 6)
Kiểu dữ liệu của bảng Price_Offer:
[String, String, Float64, Float64, Float64, Datetime(time_unit='us', time_zone=None)]
Kiểm tra số lượng dữ liệu null trong bảng Price_Offer:


offer_id,product_id,current_price,original_price,discount_percent,crawl_time
u32,u32,u32,u32,u32,u32
0,0,0,0,0,0


offer_id,product_id,current_price,original_price,discount_percent,crawl_time
str,str,f64,f64,f64,datetime[μs]
"""633304a834""","""1217527""",48000.0,48000.0,0.0,2026-02-25 19:20:12.984106
"""7d57439ca7""","""1217917""",80000.0,80000.0,0.0,2026-02-25 19:20:16.744204
"""ca2719fc3b""","""1217919""",80000.0,80000.0,0.0,2026-02-25 19:20:22.292313
"""7a2a3f51a4""","""1217923""",142000.0,142000.0,0.0,2026-02-25 19:20:28.327105
"""52890ddf91""","""1217929""",69900.0,69900.0,0.0,2026-02-25 19:20:31.262106


In [19]:
df_price_offer = df_price_offer.unique(subset=["offer_id"], keep='first') \
    .with_columns([
        pl.col("offer_id").str.strip_chars(),
        pl.col("product_id").str.strip_chars()
    ]) \
    .filter(
        pl.col("offer_id").is_not_null() & (pl.col("offer_id") != "") & 
        pl.col("product_id").is_not_null() & (pl.col("product_id") != "")
    ) \
    .with_columns([
        pl.col("product_id").cast(pl.Int64),
        pl.col("current_price").cast(pl.Float64).fill_null(0.0),
        pl.col("original_price").cast(pl.Float64).fill_null(0.0),
        pl.col("discount_percent").cast(pl.Float64).fill_null(0.0)
    ])

In [20]:
print(f"Kích thước bảng Price_Offer sau khi làm sạch: {df_price_offer.shape}")
display(df_price_offer.head())

Kích thước bảng Price_Offer sau khi làm sạch: (1027622, 6)


offer_id,product_id,current_price,original_price,discount_percent,crawl_time
str,i64,f64,f64,f64,datetime[μs]
"""739cbda092""",178477199,250000.0,250000.0,0.0,2026-02-17 11:37:25.723147
"""2c4e02a16e""",1651907,395900.0,495000.0,20.0,2026-03-01 14:30:23.888908
"""a0a524bd47""",185844624,42000.0,55000.0,24.0,2026-03-05 02:22:59.162188
"""1dfd8c9e58""",130063441,1.58e6,1.58e6,0.0,2026-03-05 11:49:39.159898
"""4a56ebc4d7""",9956739,350650.0,350650.0,0.0,2026-03-03 13:17:59.938869


<h3 id="section-5-bang-review">5. Bảng Review</h3>

Bảng `Review` được chuẩn hóa cả về nội dung văn bản lẫn ý nghĩa thời gian. Các biểu thức Regex được sử dụng để tách số lượng thời gian từ các chuỗi như `Đánh giá vào 2 năm trước` hoặc `Đã dùng 3 tháng`, sau đó quy đổi về số ngày để thuận lợi cho phân tích hành vi người dùng.

In [21]:
print(f"Kích thước bảng Review: {df_review.shape}")
print(f"Kiểu dữ liệu của bảng Review:\n{df_review.dtypes}")

print("Kiểm tra số lượng dữ liệu null trong bảng Review:")
display(df_review.null_count())
display(df_review.head())

Kích thước bảng Review: (1320038, 8)
Kiểu dữ liệu của bảng Review:
[String, String, String, Int64, String, Int64, String, String]
Kiểm tra số lượng dữ liệu null trong bảng Review:


review_id,product_id,reviewer_id,rating_score,review_content,thank_count,review_time,usage_duration
u32,u32,u32,u32,u32,u32,u32,u32
0,0,0,0,165379,0,0,4714


review_id,product_id,reviewer_id,rating_score,review_content,thank_count,review_time,usage_duration
str,str,str,i64,str,i64,str,str
"""d2f16143-fe9""","""273377046""","""679284b1e4""",5,"""""",0,"""Đánh giá vào 5 tháng trước""","""Đã dùng 2 ngày"""
"""3105eeae-88b""","""262803182""","""c9a9bab8f4""",4,"""""",0,"""Đánh giá vào 2 năm trước""","""Đã dùng 3 tháng"""
"""f0ea86b7-f9b""","""163739509""","""36138811b4""",5,"""""",0,"""Đánh giá vào 2 năm trước""","""Đã dùng 4 tháng"""
"""6cbb0f56-49c""","""163739509""","""70b76f17ab""",5,"""sách tốt, như ảnh""",0,"""Đánh giá vào 4 năm trước""","""Đã dùng 3 ngày"""
"""7446cd0a-d0e""","""163739509""","""574ea2859b""",5,"""""",0,"""Đánh giá vào 3 năm trước""","""Đã dùng 4 tháng"""


In [22]:
df_review = (
    df_review.unique(subset=["review_id"], keep='first')
    .with_columns([
        pl.col("rating_score").cast(pl.Int8),
        pl.col("thank_count").cast(pl.Int32).fill_null(0),
        
        pl.col("review_content").str.replace_all(r"\s+", " ").str.strip_chars().fill_null(""),
        
        pl.col("review_time").str.extract(r"(\d+)").cast(pl.Int32).fill_null(0).alias("time_num"),
        pl.col("review_time").str.extract(r"(năm|tháng|tuần|ngày|giờ|phút)").str.to_lowercase().fill_null("ngày").alias("time_unit"),
        
        pl.col("usage_duration").str.extract(r"(\d+)").cast(pl.Int32).fill_null(0).alias("usage_num"),
        pl.col("usage_duration").str.extract(r"(năm|tháng|tuần|ngày)").str.to_lowercase().fill_null("ngày").alias("usage_unit"),
    ])
    .with_columns([
        (
            pl.col("time_num") * pl.when(pl.col("time_unit") == "năm").then(365)
            .when(pl.col("time_unit") == "tháng").then(30)
            .when(pl.col("time_unit") == "tuần").then(7)
            .when(pl.col("time_unit").is_in(["giờ", "phút"])).then(0)
            .otherwise(1)
        ).alias("review_days_ago"),
        
        (
            pl.col("usage_num") * pl.when(pl.col("usage_unit") == "năm").then(365)
            .when(pl.col("usage_unit") == "tháng").then(30)
            .when(pl.col("usage_unit") == "tuần").then(7)
            .otherwise(1)
        ).alias("usage_days")
    ])
    .drop(["review_time", "usage_duration", "time_num", "time_unit", "usage_num", "usage_unit"])
    .sort(by=["product_id", "review_id"])
)

In [23]:
print(f"Kích thước bảng Review sau khi làm sạch: {df_review.shape}")
display(df_review.head())

Kích thước bảng Review sau khi làm sạch: (1320038, 8)


review_id,product_id,reviewer_id,rating_score,review_content,thank_count,review_days_ago,usage_days
str,str,str,i8,str,i32,i32,i32
"""1420ac0e-273""","""100023508""","""96d51f72e6""",5,"""""",0,365,2
"""359f05e9-b36""","""100023508""","""6eaff986b6""",5,"""Ok""",0,1460,8
"""672816e1-feb""","""100023508""","""6ce1f186bc""",5,"""""",0,1460,11
"""71aaffae-baf""","""100023508""","""75ff41909c""",5,"""""",0,1460,3
"""7bcf5698-ec6""","""100023508""","""1e8c004cc3""",5,"""""",0,1460,5


<h3 id="section-6-bang-reviewer">6. Bảng Reviewer</h3>

Bảng `Reviewer` được làm sạch bằng cách chuẩn hóa tên người dùng, xử lý giá trị thiếu và chuyển thông tin thâm niên từ dạng mô tả ngôn ngữ tự nhiên sang số ngày (`seniority_days`) để tăng khả năng định lượng trong phân tích.

In [24]:
print(f"Kích thước bảng Reviewer: {df_reviewer.shape}")
print(f"Kiểu dữ liệu của bảng Reviewer:\n{df_reviewer.dtypes}")

print("Kiểm tra số lượng dữ liệu null trong bảng Reviewer:")
display(df_reviewer.null_count())
display(df_reviewer.head())

Kích thước bảng Reviewer: (458141, 5)
Kiểu dữ liệu của bảng Reviewer:
[String, String, String, Int64, Int64]
Kiểm tra số lượng dữ liệu null trong bảng Reviewer:


reviewer_id,reviewer_name,reviewer_seniority,reviewer_contributions,reviewer_received_thanks
u32,u32,u32,u32,u32
0,2,2,0,0


reviewer_id,reviewer_name,reviewer_seniority,reviewer_contributions,reviewer_received_thanks
str,str,str,i64,i64
"""002304bb1b""","""Đào Quốc Thiện""","""Đã tham gia 7 năm""",21,3
"""ab814822eb""","""Quang Tuệ""","""Đã tham gia 8 năm""",64,0
"""457ec04452""","""Luân Trịnh""","""Đã tham gia 4 năm""",3,0
"""0b4eb78dff""","""Tất Liên""","""Đã tham gia 9 năm""",52,0
"""66407913dc""","""Huệ Nguyễn""","""Đã tham gia 12 năm""",1,0


In [25]:
df_reviewer = (
    df_reviewer.unique(subset=["reviewer_id"], keep='first')
    .with_columns([
        pl.col("reviewer_id").str.strip_chars(),
        pl.col("reviewer_name").str.strip_chars().fill_null("Ẩn danh"),
        pl.col("reviewer_contributions").cast(pl.Int32).fill_null(0),
        pl.col("reviewer_received_thanks").cast(pl.Int32).fill_null(0),
        
        pl.col("reviewer_seniority").str.extract(r"(\d+)").cast(pl.Int32).fill_null(0).alias("seniority_num"),
        pl.col("reviewer_seniority").str.extract(r"(năm|tháng|tuần|ngày)").str.to_lowercase().fill_null("ngày").alias("seniority_unit")
    ])
    .with_columns([
        (
            pl.col("seniority_num") * pl.when(pl.col("seniority_unit") == "năm").then(365)
            .when(pl.col("seniority_unit") == "tháng").then(30)
            .when(pl.col("seniority_unit") == "tuần").then(7)
            .otherwise(1)
        ).alias("seniority_days")
    ])
    .drop(["reviewer_seniority", "seniority_num", "seniority_unit"])
    .sort(by=["reviewer_id"])
)

In [26]:
print(f"Kích thước bảng Reviewer sau khi làm sạch: {df_reviewer.shape}")
display(df_reviewer.head())

Kích thước bảng Reviewer sau khi làm sạch: (458141, 5)


reviewer_id,reviewer_name,reviewer_contributions,reviewer_received_thanks,seniority_days
str,str,i32,i32,i32
"""000015492f""","""Rita Nguyen""",140,4,3650
"""000042f447""","""lã khánh linh""",6,1,1460
"""00008f75cb""","""Nguyễn Thị Bích Lệ""",7,2,3650
"""0000b7fbc0""","""lê thị hường""",23,2,1825
"""0000c9136b""","""Phan Ngọc Như""",32,23,2555


<h3 id="section-7-bang-service">7. Bảng Service</h3>

Bảng `Service` lưu các dịch vụ đi kèm sản phẩm/đơn hàng. Việc tiền xử lý nhằm chuẩn hóa mã dịch vụ, loại bỏ bản ghi không hợp lệ và giữ cho tập dữ liệu nhất quán trước khi liên kết sang các bảng trung gian.

In [27]:
print(f"Kích thước bảng Service: {df_service.shape}")
print(f"Kiểu dữ liệu của bảng Service:\n{df_service.dtypes}")

print("Kiểm tra số lượng dữ liệu null trong bảng Service:")
display(df_service.null_count())
display(df_service.head())

Kích thước bảng Service: (1388, 2)
Kiểu dữ liệu của bảng Service:
[Int64, String]
Kiểm tra số lượng dữ liệu null trong bảng Service:


service_id,service_name
u32,u32
0,0


service_id,service_name
i64,str
249067,"""Trả góp từ 657.500 ₫/tháng"""
249081,"""Trả góp từ 833.250 ₫/tháng"""
249086,"""Trả góp từ 383.250 ₫/tháng"""
249071,"""Trả góp từ 415.833 ₫/tháng"""
249141,"""Trả góp từ 1.523.333 ₫/tháng"""


In [28]:
df_service = (
        df_service
        .unique(subset=["service_id"], keep="first")
        
        .with_columns([
            pl.col("service_id").cast(pl.Int64),
            pl.col("service_name").str.strip_chars()
        ])
        
        .with_columns([
            pl.col("service_name")
            .str.extract(r"(\d{1,3}(?:\.\d{3})+)", 1)
            .str.replace_all(r"\.", "")
            .cast(pl.Int64, strict=False)
            .alias("installment_price_vnd")
        ])
    )

In [29]:
print(f"Kích thước bảng Service sau khi làm sạch: {df_service.shape}")
display(df_service.head())

Kích thước bảng Service sau khi làm sạch: (1388, 3)


service_id,service_name,installment_price_vnd
i64,str,i64
249360,"""Trả góp từ 418.833 ₫/tháng""",418833
249720,"""Trả góp từ 1.340.083 ₫/tháng""",1340083
250119,"""Trả góp từ 853.333 ₫/tháng""",853333
249863,"""Trả góp từ 732.666 ₫/tháng""",732666
337946,"""Trả góp từ 1.291.666 ₫/tháng""",1291666


<h3 id="section-8-bang-coupon">8. Bảng Coupon</h3>

Bảng `Coupon` được làm sạch theo hướng chuẩn hóa mã giảm giá, mô tả và các trường định danh để hỗ trợ phân tích hiệu quả khuyến mãi cũng như liên kết với bảng ưu đãi trung gian.

In [30]:
print(f"Kích thước bảng Coupon: {df_coupon.shape}")
print(f"Kiểu dữ liệu của bảng Coupon:\n{df_coupon.dtypes}")

print("Kiểm tra số lượng dữ liệu null trong bảng Coupon:")
display(df_coupon.null_count())
display(df_coupon.head())

Kích thước bảng Coupon: (1077, 5)
Kiểu dữ liệu của bảng Coupon:
[Int64, String, String, String, Date]
Kiểm tra số lượng dữ liệu null trong bảng Coupon:


coupon_id,coupon_code,title,condition,expiry
u32,u32,u32,u32,u32
0,0,0,0,0


coupon_id,coupon_code,title,condition,expiry
i64,str,str,str,date
5,"""2502TOPDEAL08""","""Giảm 20K""","""Cho đơn hàng từ 299K""",2026-02-25
5321,"""2502TOPDEAL03""","""Giảm 8% tối đa 50K""","""Cho đơn hàng từ 499K""",2026-02-25
295,"""THO8688""","""Giảm 5% tối đa 30K""","""Cho đơn hàng từ 500K""",2026-02-28
294,"""THBO88""","""Giảm 5% tối đa 15K""","""Cho đơn hàng từ 299K""",2026-02-28
8838,"""VCVIP11K""","""Giảm 11K""","""Cho đơn hàng từ 1.4 triệu""",2026-12-31


In [31]:
df_coupon = (
        df_coupon
        .unique(subset=["coupon_id"], keep="first")
        .with_columns([
            pl.col("coupon_id").cast(pl.Int64),
            pl.col("coupon_code").str.strip_chars().fill_null("NO_CODE"),
            pl.col("title").str.strip_chars(),
            pl.col("condition").str.strip_chars()
        ])
        .with_columns([
            pl.col("expiry").cast(pl.String).str.to_date("%Y-%m-%d", strict=False)
        ])
    )

In [32]:
print(f"Kích thước bảng Coupon sau khi làm sạch: {df_coupon.shape}")
display(df_coupon.head())

Kích thước bảng Coupon sau khi làm sạch: (1077, 5)


coupon_id,coupon_code,title,condition,expiry
i64,str,str,str,date
3,"""2502TOPDEAL02""","""Giảm 8% tối đa 50K""","""Cho đơn hàng từ 299K""",2026-02-25
17870,"""20KNOLULU""","""Giảm 20K""","""Cho đơn hàng từ 500K""",2026-12-31
385776,"""VC90K202""","""Giảm 90K""","""Cho đơn hàng từ 820K""",2026-05-31
122444,"""0303TOPDEAL04""","""Giảm 8% tối đa 100K""","""Cho đơn hàng từ 699K""",2026-03-03
362312,"""LOCK5ALLT0326A""","""Giảm 5% tối đa 100K""","""Cho đơn hàng từ 100K""",2026-03-17


<h3 id="section-9-bang-offer-coupon">9. Bảng Offer Coupon</h3>

Đây là bảng liên kết nhiều-nhiều giữa `Price Offer` và `Coupon`. Quá trình làm sạch tập trung vào loại bỏ trùng lặp theo cặp khóa, chuẩn hóa định danh và đảm bảo mỗi liên kết đều có ý nghĩa phân tích.

In [33]:
print(f"Kích thước bảng Offer_Coupon: {df_offer_coupon.shape}")
print(f"Kiểu dữ liệu của bảng Offer_Coupon:\n{df_offer_coupon.dtypes}")

print("Kiểm tra số lượng dữ liệu null trong bảng Offer_Coupon:")
display(df_offer_coupon.null_count())
display(df_offer_coupon.head())

Kích thước bảng Offer_Coupon: (1519260, 2)
Kiểu dữ liệu của bảng Offer_Coupon:
[String, Int64]
Kiểm tra số lượng dữ liệu null trong bảng Offer_Coupon:


offer_id,coupon_id
u32,u32
0,0


offer_id,coupon_id
str,i64
"""f4cb55c547""",1
"""f4cb55c547""",2
"""f4cb55c547""",3
"""f4cb55c547""",4
"""f4cb55c547""",5


In [34]:
df_offer_coupon = df_offer_coupon.unique(subset=["offer_id", "coupon_id"], keep='first') \
    .with_columns([
        pl.col("offer_id").str.strip_chars(),
        pl.col("coupon_id").cast(pl.Int64)
    ]) \
    .drop_nulls() \
    .filter(
        pl.col("offer_id") != ""
    )

In [35]:
print(f"Kích thước bảng Offer_Coupon sau khi làm sạch: {df_offer_coupon.shape}")
display(df_offer_coupon.head())

Kích thước bảng Offer_Coupon sau khi làm sạch: (1519260, 2)


offer_id,coupon_id
str,i64
"""9ec79626a6""",71
"""401460c129""",11
"""23327f4549""",352855
"""aeed6f156f""",352842
"""3284544d82""",26658


<h3 id="section-10-bang-offer-service">10. Bảng Offer Service</h3>

Tương tự `Offer Coupon`, bảng `Offer Service` đóng vai trò trung gian để biểu diễn các dịch vụ áp dụng cho từng offer. Dữ liệu được chuẩn hóa khóa và loại bỏ bản ghi rỗng trước khi phục vụ bước kiểm tra liên kết.

In [36]:
print(f"Kích thước bảng Offer_Service: {df_offer_service.shape}")
print(f"Kiểu dữ liệu của bảng Offer_Service:\n{df_offer_service.dtypes}")

print("Kiểm tra số lượng dữ liệu null trong bảng Offer_Service:")
display(df_offer_service.null_count())
display(df_offer_service.head())

Kích thước bảng Offer_Service: (944484, 2)
Kiểu dữ liệu của bảng Offer_Service:
[String, Int64]
Kiểm tra số lượng dữ liệu null trong bảng Offer_Service:


offer_id,service_id
u32,u32
0,0


offer_id,service_id
str,i64
"""282f24cadf""",1
"""282f24cadf""",2
"""f4cb55c547""",1
"""f4cb55c547""",2
"""8c55765cab""",1


In [37]:
df_offer_service = df_offer_service.unique(subset=["offer_id", "service_id"], keep='first') \
    .with_columns([
        pl.col("offer_id").str.strip_chars(),
        pl.col("service_id").cast(pl.Int64)
    ]) \
    .drop_nulls() \
    .filter(
        pl.col("offer_id") != ""
    )

In [38]:
print(f"Kích thước bảng Offer_Service sau khi làm sạch: {df_offer_service.shape}")
display(df_offer_service.head())

Kích thước bảng Offer_Service sau khi làm sạch: (944484, 2)


offer_id,service_id
str,i64
"""79ecd5eae2""",2
"""9e39163eaa""",1
"""52491eba2a""",2
"""299d74c055""",2
"""24c06881d4""",2


<h2 id="luu-tru-du-lieu">LƯU TRỮ DỮ LIỆU</h2>

Dữ liệu sau khi làm sạch được lưu dưới định dạng `.parquet` để phục vụ các bước phân tích tiếp theo.

**Lý do chọn Parquet:**
- Giữ nguyên schema và kiểu dữ liệu sau tiền xử lý.
- Dung lượng nén thường nhỏ hơn đáng kể so với `.csv`.
- Tốc độ đọc/ghi tối ưu hơn cho khối lượng dữ liệu lớn và các pipeline phân tích dữ liệu.

In [39]:
PROCESSED_PATH = "../data/processed"

df_category.write_parquet(f"{PROCESSED_PATH}/category.parquet")
df_seller.write_parquet(f"{PROCESSED_PATH}/seller.parquet")
df_product.write_parquet(f"{PROCESSED_PATH}/product.parquet")
df_price_offer.write_parquet(f"{PROCESSED_PATH}/price_offer.parquet")
df_review.write_parquet(f"{PROCESSED_PATH}/review.parquet")
df_reviewer.write_parquet(f"{PROCESSED_PATH}/reviewer.parquet")
df_service.write_parquet(f"{PROCESSED_PATH}/service.parquet")
df_coupon.write_parquet(f"{PROCESSED_PATH}/coupon.parquet")
df_offer_coupon.write_parquet(f"{PROCESSED_PATH}/offer_coupon.parquet")
df_offer_service.write_parquet(f"{PROCESSED_PATH}/offer_service.parquet")